# Обучение ODIN на NBV Stage3 Dataset (Multi-object + Obstacles, 3 Textures)
Этот ноутбук обучает ODIN на датасете NBV Stage 3 с препятствиями (8 геометрических форм × 3 текстуры = 24 класса + препятствия).

## Информация о датасете

**NBV Stage 3 Dataset**: https://www.kaggle.com/datasets/sergkurchevusa/nbv-stage-3-dataset-multi-object-obstacles-correct

### Характеристики:
- **Objects**: 2-10 примитивов (8 классов) + препятствия
- **Textures**: 3 типа (red, mixed, green)
- **Total Classes**: 24 (8 shapes × 3 textures)
- **Images**: 224×224 RGB + depth + masks
- **Cameras**: **5 кадров** на сцену

In [ ]:
import os
import subprocess
import sys
import urllib.request

# Базовая конфигурация проекта
CONFIG = {
    "ODIN_DIR": "my_odin",
    "ODIN_REPO_URL": "https://github.com/SergKurchev/my_odin.git",
    "RL_DIR": "article-nbv",
    "RL_REPO_URL": "https://github.com/SergKurchev/article-nbv.git",
    # Публичные веса из katefgroup/odin
    "ODIN_WEIGHTS_URL": "https://huggingface.co/katefgroup/odin/resolve/main/scannet_resnet_47.8_73.3_32k_1.5k.pth",
    "ODIN_WEIGHTS_PATH": "my_odin/models/odin_scannet_context.pth",
    "M2F_WEIGHTS_URL": "https://huggingface.co/katefgroup/odin/resolve/main/m2f_coco.pkl",
    "M2F_WEIGHTS_PATH": "my_odin/models/model_final_5c90d4.pkl",
}


## 1. Создание виртуального окружения и установка зависимостей

In [ ]:
# 1. Создаем venv (даже если ensurepip нет, структура папок создастся)
if not os.path.exists("venv"):
    print("Устанавливаю Python 3.10...")
    subprocess.run(["apt-get", "update", "-y"], check=False)
    subprocess.run(["apt-get", "install", "-y", "python3.10", "python3.10-venv", "python3.10-dev", "python3.10-distutils"], check=False)

    print("Создаю виртуальное окружение на Python 3.10...")
    # Флаг --without-pip предотвращает попытки использовать отсутствующий ensurepip
    subprocess.run(["python3.10", "-m", "venv", "venv", "--without-pip"], check=True)

    VENV_PYTHON_TMP = os.path.abspath("venv/bin/python")

    # Прямая установка pip через официальный скрипт
    print("Загружаю get-pip.py для ручной установки...")
    urllib.request.urlretrieve("https://bootstrap.pypa.io/get-pip.py", "get-pip.py")

    print("Устанавливаю pip в виртуальное окружение...")
    subprocess.run([VENV_PYTHON_TMP, "get-pip.py"], check=True)

    # Чистим за собой
    if os.path.exists("get-pip.py"):
        os.remove("get-pip.py")
    print("Pip успешно установлен!")

# Теперь пути будут корректными
VENV_PYTHON = os.path.abspath("venv/bin/python")
VENV_PIP = os.path.abspath("venv/bin/pip")


def make_venv_env(extra=None):
    """
    Создаёт словарь переменных окружения с правильными путями к venv.

    КЛЮЧЕВОЕ ОТЛИЧИЕ от системной среды (reference):
    В системной среде pip/python берутся из PATH и уже знают о системных пакетах.
    В venv — нужно явно прописать VIRTUAL_ENV и PATH, иначе subprocess-ы,
    запущенные из подпроцессов (например, build isolation в pip),
    не будут знать, что они внутри venv и не найдут torch.
    """
    env = os.environ.copy()
    venv_dir = os.path.abspath("venv")
    env["VIRTUAL_ENV"] = venv_dir
    env["PATH"] = os.path.join(venv_dir, "bin") + ":" + env.get("PATH", "")
    # Убираем PYTHONPATH системы — он может мешать изоляции
    env.pop("PYTHONPATH", None)
    if extra:
        env.update(extra)
    return env


def clean_build_artifacts(directory):
    """
    Очищает старые артефакты сборки перед компиляцией.

    Ref-функция это делает явно — без очистки повторный запуск
    может завершиться ошибкой из-за конфликта старых .egg файлов.
    """
    import shutil
    import glob
    for d in ["build", "dist"]:
        path = os.path.join(directory, d)
        if os.path.exists(path):
            print(f"   Cleaning {path}...")
            shutil.rmtree(path)
    for egg in glob.glob(os.path.join(directory, "*.egg-info")):
        print(f"   Cleaning {egg}...")
        shutil.rmtree(egg)


def run_in_venv(cmd, cwd=None, env=None, check=True):
    """Обертка для запуска команд внутри созданного виртуального окружения."""
    if cmd[0] == "pip":
        cmd[0] = VENV_PIP
    elif cmd[0] == "python":
        cmd[0] = VENV_PYTHON

    print(f"Выполняю: {' '.join(cmd)}")
    subprocess.run(cmd, cwd=cwd, env=env, check=check)


def install_odin_dependencies():
    """Установка зависимостей ODIN в виртуальное окружение."""
    print("\n" + "=" * 80)
    print("Installing ODIN Dependencies into VENV")
    print("=" * 80)

    venv_env = make_venv_env()

    # 0. Клонирование репозитория
    if not os.path.exists(CONFIG["ODIN_DIR"]):
        print("\n0. Cloning ODIN...")
        subprocess.run(["git", "clone", "-q", "-b", "feature/nbv_dataset_process", CONFIG["ODIN_REPO_URL"], CONFIG["ODIN_DIR"]], check=True)

    # 0b. Клонирование репозитория article-nbv для визуализаций
    if not os.path.exists(CONFIG["RL_DIR"]):
        print("\n0b. Cloning article-nbv...")
        subprocess.run(["git", "clone", "-q", CONFIG["RL_REPO_URL"], CONFIG["RL_DIR"]], check=True)

    # 1. PyTorch 2.2.0 + CUDA 12.1
    print("\n1. Installing PyTorch 2.2.0...")
    run_in_venv([
        "pip", "install", "-q", "torch==2.2.0", "torchvision==0.17.0",
        "--index-url", "https://download.pytorch.org/whl/cu121"
    ], env=venv_env)

    run_in_venv([
        "pip", "install", "-q", "torch-scatter",
        "-f", "https://data.pyg.org/whl/torch-2.2.0+cu121.html"
    ], env=venv_env)

    # 2. NumPy < 2 и Pillow
    print("\n2. Installing NumPy < 2 + Pillow...")
    run_in_venv(["pip", "install", "-q", "numpy<2", "--force-reinstall"], env=venv_env)
    run_in_venv(["pip", "install", "-q", "Pillow>=10.2.0"], env=venv_env)

    # 3. Clean requirements (через Python — надежнее чем sed)
    print("\n3. Cleaning ODIN requirements...")
    req_path = os.path.join(CONFIG["ODIN_DIR"], "requirements.txt")

    with open(req_path, 'r') as f:
        lines = f.readlines()

    with open(req_path, 'w') as f:
        for line in lines:
            line_clean = line.strip().lower()
            # Пропускаем все проблемные пакеты
            if any(x in line_clean for x in ["waspinator", "detectron2", "pytorch3d"]):
                print(f"   Skipping: {line.strip()}")
                continue
            # Фикс для старого PyYAML
            if "pyyaml==5.3.1" in line_clean:
                f.write("pyyaml>=5.4.1\n")
            else:
                f.write(line)

    # 4. Build tools + Modern COCO API (нужны перед requirements.txt)
    print("\n4. Installing Build Tools & Modern dependencies...")
    run_in_venv(["pip", "install", "-q", "cython", "setuptools", "wheel", "pycocotools"], env=venv_env)

    # 5. ODIN requirements + ninja, fvcore, iopath
    print("\n5. Installing ODIN requirements...")
    run_in_venv(["pip", "install", "-q", "-r", req_path], env=venv_env)
    run_in_venv(["pip", "install", "-q", "ninja", "fvcore", "iopath"], env=venv_env)

    # 6. Detectron2
    # КЛЮЧЕВОЕ ОТЛИЧИЕ от системной среды:
    # В системной среде pip видит torch в системных пакетах Kaggle.
    # В нашем venv pip должен использовать torch из venv,
    # поэтому нужен --no-build-isolation (запрет временного пустого build env)
    # + явно прокидываем venv_env с правильным VIRTUAL_ENV и PATH.
    print("\n6. Installing Detectron2...")
    run_in_venv([
        "pip", "install", "-q", "--no-build-isolation",
        "git+https://github.com/facebookresearch/detectron2.git"
    ], env=venv_env)

    # 7. PyTorch3D
    print("\n7. Installing PyTorch3D (with CUDA)...")
    cuda_env = make_venv_env({
        "FORCE_CUDA": "1",
        "TORCH_CUDA_ARCH_LIST": "6.0;7.0;7.5;8.0;8.6",
    })
    run_in_venv([
        "pip", "install", "-q", "--no-build-isolation",
        "git+https://github.com/facebookresearch/pytorch3d.git"
    ], env=cuda_env)

    # 8. Критически важно: переустановка правильных версий NumPy и OpenCV
    # (другие пакеты могут их обновить в процессе установки)
    print("\n8. Re-installing correct NumPy + OpenCV...")
    run_in_venv(["pip", "uninstall", "-y", "-q", "numpy"], env=venv_env)
    run_in_venv(["pip", "install", "-q", "numpy==1.26.4"], env=venv_env)
    run_in_venv(["pip", "install", "-q", "opencv-python-headless==4.8.0.76"], env=venv_env)

    # 9. Компиляция CUDA-ядер pointops2
    # КЛЮЧЕВОЕ ОТЛИЧИЕ:
    # Ref-функция делает os.chdir() + setup.py install --user.
    # --user ставит в ~/.local, откуда системный python видит, но venv — нет!
    # Мы передаём cwd= и VENV_PYTHON напрямую, БЕЗ --user.
    # Очистка артефактов — как в ref-функции.
    print("\n9. Compiling CUDA kernels (pointops2)...")
    pointops_dir = os.path.abspath(os.path.join(CONFIG["ODIN_DIR"], "libs", "pointops2"))
    clean_build_artifacts(pointops_dir)
    run_in_venv(["python", "setup.py", "install"], cwd=pointops_dir, env=cuda_env)

    # 10. Компиляция CUDA-ядер deformable attention
    print("\n10. Compiling CUDA kernels (deformable attention)...")
    deform_dir = os.path.abspath(os.path.join(CONFIG["ODIN_DIR"], "odin", "modeling", "pixel_decoder", "ops"))
    clean_build_artifacts(deform_dir)
    run_in_venv(["python", "setup.py", "build", "install"], cwd=deform_dir, env=cuda_env)

    print("\n" + "=" * 80)
    print("Installation complete!")
    print("=" * 80)


def download_weights():
    """Скачивание предобученных весов с публичных URL (katefgroup/odin)."""
    print("\n" + "=" * 80)
    print("Downloading Weights")
    print("=" * 80)

    os.makedirs(os.path.dirname(CONFIG["ODIN_WEIGHTS_PATH"]), exist_ok=True)

    def download_file(url, dest_path, min_size_mb=50):
        """Скачивает файл и проверяет целостность по размеру."""
        size_mb = os.path.getsize(dest_path) / (1024 * 1024) if os.path.exists(dest_path) else 0
        if size_mb >= min_size_mb:
            print(f"   ✓ Файл уже скачан ({size_mb:.0f} MB): {dest_path}")
            return
        if os.path.exists(dest_path):
            print(f"   ⚠️  Файл повреждён ({size_mb:.1f} MB), удаляю...")
            os.remove(dest_path)
        print(f"   Скачиваю {url} ...")
        ret = os.system(f"wget --tries=3 --retry-connrefused -c '{url}' -O '{dest_path}'")
        if ret != 0:
            print(f"   ❌ wget завершился с кодом {ret}, пробую curl...")
            os.system(f"curl -L --retry 3 '{url}' -o '{dest_path}'")
        size_mb = os.path.getsize(dest_path) / (1024 * 1024) if os.path.exists(dest_path) else 0
        print(f"   ✓ Скачано: {dest_path} ({size_mb:.0f} MB)")
        if size_mb < min_size_mb:
            raise RuntimeError(f"Файл слишком мал ({size_mb:.1f} MB) — скачивание провалилось!")

    print("\n1. Downloading ODIN weights (ScanNet ResNet50)...")
    download_file(CONFIG["ODIN_WEIGHTS_URL"], CONFIG["ODIN_WEIGHTS_PATH"], min_size_mb=100)

    print("\n2. Downloading Mask2Former weights (M2F COCO ResNet)...")
    download_file(CONFIG["M2F_WEIGHTS_URL"], CONFIG["M2F_WEIGHTS_PATH"], min_size_mb=50)

    print("\nWeights ready!")


# Запускаем сборку
install_odin_dependencies()
download_weights()

## 2. Диагностика датасета и создание splits

NBV Stage 3 датасет имеет следующую структуру (проверено локально):

In [ ]:
# Диагностика NBV Stage 3 датасета и создание splits
import os
import json
from pathlib import Path

# Автоматический поиск датасета
DATASET_DIR = None
possible_paths = [
    "/kaggle/input/datasets/sergkurchevusa/nbv-stage-3-dataset-multi-object-obstacles-correct/stage3",
    "/kaggle/input/nbv-stage-3-dataset-multi-object-obstacles-correct/stage3",
    "/kaggle/input/nbv-stage-3-dataset-multi-object-obstacles-correct",
    "/kaggle/input/datasets/sergkurchevusa/nbv-stage-3-dataset-multi-object-obstacles-correct",
    "./dataset/primitives/stage3"
]
for p in possible_paths:
    if os.path.exists(p):
        try:
            if any(e.startswith("sample_") for e in os.listdir(p)):
                DATASET_DIR = p
                print(f"✓ Found dataset directory: {p}")
                break
        except:
            pass

if DATASET_DIR is None:
    if os.path.exists("/kaggle/input"):
        for root, dirs, files in os.walk("/kaggle/input"):
            if "stage3" in root or "stage_3" in root:
                if any(d.startswith("sample_") for d in dirs):
                    DATASET_DIR = root
                    print(f"✓ Autodetected dataset directory: {DATASET_DIR}")
                    break

if DATASET_DIR is None:
    DATASET_DIR = "/kaggle/input/datasets/sergkurchevusa/nbv-stage-3-dataset-multi-object-obstacles-correct/stage3"

SPLITS_FILE = "./nbv_stage3_splits.json"

print("=== Диагностика NBV Stage 3 датасета ===")
print(f"DATASET_DIR: {DATASET_DIR}")
print(f"DATASET_DIR exists: {os.path.exists(DATASET_DIR)}")

if os.path.exists(DATASET_DIR):
    entries = sorted([e for e in os.listdir(DATASET_DIR) if e.startswith("sample_")])
    print(f"\nНайдено {len(entries)} сэмплов")
    print(f"Первые 10: {entries[:10]}")
    print(f"Последние 10: {entries[-10:]}")

    import random
    random.seed(42)

    n_samples = len(entries)
    n_train = int(0.8 * n_samples)
    n_val = int(0.1 * n_samples)

    shuffled_entries = entries.copy()
    random.shuffle(shuffled_entries)

    splits = {
        "train": shuffled_entries[:n_train],
        "val": shuffled_entries[n_train:n_train + n_val],
        "test": shuffled_entries[n_train + n_val:]
    }

    with open(SPLITS_FILE, 'w') as f:
        json.dump(splits, f, indent=2)

    print(f"\n=== Созданы splits (random_state=42) ===")
    for split_name, ids in splits.items():
        print(f"{split_name}: {len(ids)} сэмплов")

    # Проверим структуру одного сэмпла
    sample_dir = Path(DATASET_DIR) / entries[0]
    if sample_dir.exists():
        print(f"\n=== Структура {entries[0]} ===")
        for item in sorted(sample_dir.iterdir()):
            if item.is_dir():
                count = len(list(item.iterdir()))
                print(f"  {item.name}/ ({count} файлов)")
            else:
                print(f"  {item.name}")


## 3. Запуск обучения с SWAG

### Особенности датасета NBV Stage3 (vs Stage 2):
- **24 класса + препятствия**: 8 примитивов × 3 текстуры
- **Наличие препятствий**: Stage 3 сложнее, так как присутствуют перекрытия
- **5 кадров** на сэмпл

> **Важно**: `--num_frames 5` — это максимум. Можно уменьшить для экономии VRAM.

In [ ]:
import shutil

def sync_previous_output():
    """Копирует старые чекпоинты для --resume"""
    input_base = "/kaggle/input"
    target_output = "./output_nbv_stage3_active"
    
    if not os.path.exists(input_base):
        return

    for root, dirs, files in os.walk(input_base):
        if ("output_nbv_stage3_active" in root or "output" in root) and any(f.endswith(".pth") for f in files):
            print(f"Нашел старые чекпоинты в: {root}")
            if not os.path.exists(target_output):
                os.makedirs(target_output)
            
            for f in files:
                src = os.path.join(root, f)
                dst = os.path.join(target_output, f)
                if not os.path.exists(dst):
                    shutil.copy2(src, dst)
            print(f"Успешно скопировал {len(files)} файлов в {target_output}")
            return

sync_previous_output()


In [ ]:
run_in_venv([
        "git", "pull"
    ], cwd = "/kaggle/working/my_odin")

In [ ]:
run_in_venv([
        "git", "rev-parse", "--short", "HEAD"
    ], cwd = "/kaggle/working/my_odin")

In [ ]:
# Используем DATASET_DIR и SPLITS_FILE для Stage 3
CONFIG_FILE = "my_odin/configs/scannet_context/3d.yaml"

# Ищем начальные веса (по приоритету):
# 1. Чекпоинт текущего запуска Stage 3 (для --resume)
# 2. Обученные веса Stage 2 active
# 3. Базовые веса ScanNet
INITIAL_WEIGHTS = CONFIG["ODIN_WEIGHTS_PATH"]

possible_weights = [
    "./output_nbv_stage3_active/model_final.pth",
    "./output_nbv_stage3_active/model_best.pth",
    "/kaggle/input/strawpick-segpoinnet-my-odin-nbv-2-active/output_nbv_stage2_active/model_final.pth",
    "/kaggle/input/strawpick-segpoinnet-my-odin-nbv-2-active/model_final.pth",
    "/kaggle/input/strawpick-segpoinnet-my-odin-nbv-2-active/output_nbv_stage2_active/model_best.pth",
]

for w in possible_weights:
    if os.path.exists(w):
        INITIAL_WEIGHTS = w
        print(f"✓ Found initial weights: {w}")
        break

train_cmd = [
    VENV_PYTHON, "my_odin/my_train_odin.py",
    "--config-file", CONFIG_FILE,
    "--num-gpus", "1",
    "--dist-url", "tcp://127.0.0.1:23456",
    "--resume",
    "--visualize",
    "--dataset_dir", DATASET_DIR,
    "--splits_file", SPLITS_FILE,
    
    # ПАРАМЕТРЫ ОБУЧЕНИЯ
    "--num_epochs", "50",
    "--eval_period", "200",
    "--checkpoint_period", "200",
    
    # ОГРАНИЧЕНИЕ ВРЕМЕНИ
    "--max_time", "6",
    
    # ПАРАМЕТРЫ РЕСУРСОВ
    "--image_size", "224",
    "--batch_size", "3",
    "--num_frames", "5",
    "--lr", "0.00005",
    "--warmup_ratio", "0.3",
    
    # ТЕХНИЧЕСКИЕ ПАРАМЕТРЫ
    'MODEL.WEIGHTS', INITIAL_WEIGHTS,
    'OUTPUT_DIR', './output_nbv_stage3_active',
    'SOLVER.AMP.ENABLED', 'True',
    
    # SWAG ПАРАМЕТРЫ
    'MODEL.NBV_ACTIVE', 'True',
    'MODEL.COVERAGE_HEAD.LOSS_WEIGHT', '1.0',
    'MODEL.NBV_HEAD.LOSS_WEIGHT', '0.5',
    'MODEL.BAYESIAN_TYPE', 'swag',
    'MODEL.BAYESIAN_SAMPLES', '1',
    'MODEL.BAYESIAN_INFERENCE_DURING_TRAINING', 'False',
    'MODEL.SWAG.UPDATE_FREQ', '5',
    'MODEL.SWAG.MAX_MODELS', '10',
    'MODEL.SWAG.RANK', '20',
    'MODEL.SWAG.NO_COV_MAT', 'False',
]

print("Starting SWAG training on NBV Stage3 dataset...")
print(f"Initial weights: {INITIAL_WEIGHTS}")
print("Dataset: Multi-object (2-10) + Obstacles with 3 textures")

venv_env = make_venv_env()
run_in_venv(train_cmd, env=venv_env)


## 4. Inference с SWAG (после обучения)

Запустите эту ячейку после завершения обучения для полного Bayesian inference с uncertainty estimation.

In [ ]:
# Ищем финальную обученную модель для inference
FINAL_MODEL = './output_nbv_stage3_active/model_final.pth'
if not os.path.exists(FINAL_MODEL):
    if os.path.exists('./output_nbv_stage3_active/model_best.pth'):
        FINAL_MODEL = './output_nbv_stage3_active/model_best.pth'

inference_cmd = [
    VENV_PYTHON, "my_odin/my_train_odin.py",
    "--config-file", CONFIG_FILE,
    "--num-gpus", "1",
    "--eval-only",
    "--dataset_dir", DATASET_DIR,
    "--splits_file", SPLITS_FILE,
    
    'MODEL.WEIGHTS', FINAL_MODEL,
    'OUTPUT_DIR', './output_nbv_stage3_active',
    
    # SWAG INFERENCE
    'MODEL.NBV_ACTIVE', 'True',
    'MODEL.COVERAGE_HEAD.LOSS_WEIGHT', '1.0',
    'MODEL.NBV_HEAD.LOSS_WEIGHT', '0.5',
    'MODEL.BAYESIAN_TYPE', 'swag',
    'MODEL.BAYESIAN_SAMPLES', '10',
    'MODEL.SWAG.SCALE', '1.0',
]

print(f"Running SWAG inference using weights: {FINAL_MODEL}...")
venv_env = make_venv_env()
run_in_venv(inference_cmd, env=venv_env)


## Результаты

Метрики сохраняются в:
- `output_nbv_stage3_active/metrics_comparison.csv`
- `output_nbv_stage3_active/swag_state.pth` - SWAG статистика
- `output_nbv_stage3_active/model_final.pth` - финальная модель


In [ ]:
# 5. Генерация 3D визуализаций (HTML point clouds)
import subprocess

VIS_OUT_DIR = "./output_nbv_stage3_active/visualizations"
print("Генерирую 3D визуализации для первых 5 сэмплов Stage 3...")

vis_cmd = [
    VENV_PYTHON, "article-nbv/scripts/generate_all_visualizations.py",
    "--stage", "3",
    "--max-samples", "5",
    "--dataset-dir", DATASET_DIR,
    "--out-dir", VIS_OUT_DIR,
    "--python", VENV_PYTHON
]

subprocess.run(vis_cmd, check=True)
print(f"Визуализации успешно сохранены в: {VIS_OUT_DIR}")


In [ ]:
# 6. Упаковка результатов в ZIP
import shutil
from pathlib import Path

print("Упаковываю результаты обучения и визуализации...")
results_dir = Path("kaggle_results")
results_dir.mkdir(exist_ok=True)

# Копируем чекпоинты и визуализации
output_dir = Path("./output_nbv_stage3_active")
if output_dir.exists():
    shutil.copytree(output_dir, results_dir / "output_nbv_stage3_active", dirs_exist_ok=True)

# Создаем ZIP-архив
shutil.make_archive("kaggle_results_stage3", "zip", results_dir)
print("Результаты успешно упакованы в: kaggle_results_stage3.zip")
